# A Time Ordered Public Return Experiment

## Enforcing the information boundary in public data

For industry $i$ and decision date $t$, the feature vector is

$$
x_{i,t}
=
\left(
1,
\overline r_{i,t}^{(5)},
\overline r_{i,t}^{(20)},
\overline r_{i,t}^{(60)},
s_{i,t}^{(5)},
s_{i,t}^{(20)},
s_{i,t}^{(60)}
\right)^{\mathsf T}.
$$

Here, $\overline r_{i,t}^{(h)}$ and $s_{i,t}^{(h)}$ are the mean and sample standard deviation of the previous $h$ daily percentage returns. Every window ends on the day before $t$. The leading one is an intercept. Standardization uses the training period only.

### The Supplied Portfolio Vertices

Each feasible decision holds exactly four of the sixteen industries with equal weights:

$$
V
=
\left\{
w\in\{0,1/4\}^{16}:
\sum_{i=1}^{16}w_i=1
\right\}.
$$

The set contains $\binom{16}{4}=1{,}820$ vertices. The observed daily cost vector is $c_t=-r_t$, so minimizing cost is the same as maximizing the next observed portfolio return. This is a finite teaching experiment, not a trading recommendation.

In [1]:
from io import StringIO
from itertools import combinations
from pathlib import Path
from zipfile import ZipFile
import sys
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display as _display

def display(value):
    module = value.__class__.__module__
    if module.startswith('pandas') and hasattr(value, '_repr_html_'):
        html = value._repr_html_()
        return _display(HTML('<div style="display:flex;justify-content:center;">' + html + '</div>'))
    return _display(value)
BOOK_ROOT = Path.cwd().parents[1]
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))
from book_support import ensure_dataset
import RiskLabAI
from RiskLabAI.optimization import robust_mean_variance, spo_plus_loss, spo_plus_torch
assert RiskLabAI.__version__ == '3.2.0'
DATA_PATH = ensure_dataset(BOOK_ROOT, 'french_49_industries')
FIGURES = BOOK_ROOT / 'outputs' / 'figures' / '27_decision_focused_learning'
TABLES = BOOK_ROOT / 'outputs' / 'tables' / '27_decision_focused_learning'
FIGURE_DATA = BOOK_ROOT / 'outputs' / 'figure_data' / '27_decision_focused_learning'
for directory in [FIGURES, TABLES, FIGURE_DATA]:
    directory.mkdir(parents=True, exist_ok=True)
PALETTE = {'background': '#111827', 'axes': '#111827', 'text': '#E5E7EB', 'cyan': '#38BDF8', 'amber': '#FBBF24', 'green': '#34D399', 'rose': '#FB7185', 'violet': '#C084FC', 'grid': '#374151'}
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 11.5, 'figure.facecolor': PALETTE['background'], 'axes.facecolor': PALETTE['axes'], 'axes.edgecolor': PALETTE['text'], 'axes.labelcolor': PALETTE['text'], 'axes.titlecolor': PALETTE['text'], 'xtick.color': PALETTE['text'], 'ytick.color': PALETTE['text'], 'text.color': PALETTE['text'], 'grid.color': PALETTE['grid'], 'grid.linestyle': ':', 'legend.facecolor': PALETTE['axes'], 'legend.edgecolor': PALETTE['grid'], 'savefig.facecolor': PALETTE['background'], 'savefig.transparent': False, 'savefig.dpi': 300})

def rendered_value(value):
    if pd.isna(value):
        return ''
    if isinstance(value, (bool, np.bool_)):
        return 'Pass' if bool(value) else 'Fail'
    if isinstance(value, (pd.Timestamp, np.datetime64)):
        return pd.Timestamp(value).date().isoformat()
    if isinstance(value, (int, np.integer)):
        return f'{int(value):,}'
    if isinstance(value, (float, np.floating)):
        number = float(value)
        if number == 0:
            return '0'
        if number.is_integer():
            return f'{int(number):,}'
        magnitude = abs(number)
        if magnitude < 0.0001 or magnitude >= 1000000:
            return f'{number:.6g}'
        return f'{number:.6f}'.rstrip('0').rstrip('.')
    return str(value)

def rendered_frame(frame):
    return frame.apply(lambda column: column.map(rendered_value))

def save_table(frame, stem, caption, label, fit=False):
    shown = rendered_frame(frame)
    shown.to_csv(TABLES / f'{stem}.csv', index=False)
    latex = shown.to_latex(index=False, escape=True, caption=caption, label=label, position='htbp')
    if fit:
        latex = latex.replace('\\begin{tabular}', '\\begin{adjustbox}{max width=\\textwidth}\\begin{tabular}', 1).replace('\\end{tabular}', '\\end{tabular}\\end{adjustbox}', 1)
    (TABLES / f'{stem}.tex').write_text(latex, encoding='utf-8')
    display(shown)
    return shown

def finish_axis(axis, grid=True):
    axis.spines['top'].set_visible(False)
    axis.spines['right'].set_visible(False)
    if grid:
        axis.grid(True, alpha=0.65)
    axis.set_axisbelow(True)

def save_figure(figure, stem):
    figure.savefig(FIGURES / f'{stem}.png', dpi=300, bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    figure.savefig(FIGURES / f'{stem}.pdf', bbox_inches='tight', facecolor=PALETTE['background'], transparent=False)
    display(figure)

def load_public_returns():
    assert DATA_PATH.is_file()
    with ZipFile(DATA_PATH) as archive:
        member = archive.namelist()[0]
        lines = archive.read(member).decode('latin1').splitlines()
    title_index = next((index for index, line in enumerate(lines) if 'Average Value Weighted Returns -- Daily' in line))
    header_index = title_index + 1
    end_index = next((index for index in range(header_index + 1, len(lines)) if not lines[index].strip()))
    frame = pd.read_csv(StringIO('\n'.join(lines[header_index:end_index])))
    frame = frame.rename(columns={frame.columns[0]: 'Date'})
    frame['Date'] = pd.to_datetime(frame['Date'].astype(str), format='%Y%m%d')
    industries = list(frame.columns[1:17])
    frame[industries] = frame[industries].apply(pd.to_numeric, errors='coerce')
    frame[industries] = frame[industries].mask(frame[industries] <= -99)
    frame = frame.loc[(frame['Date'] >= '2019-01-01') & (frame['Date'] <= '2025-12-31'), ['Date', *industries]].dropna().reset_index(drop=True)
    assert len(industries) == 16
    assert len(frame) == 1760
    return (frame, industries)
import torch
public, industries = load_public_returns()
returns = public.set_index('Date')[industries]
feature_parts = []
feature_names = []
for window in [5, 20, 60]:
    feature_parts.append(returns.rolling(window).mean().shift(1))
    feature_names.append(f'Past {window}-day mean')
for window in [5, 20, 60]:
    feature_parts.append(returns.rolling(window).std(ddof=1).shift(1))
    feature_names.append(f'Past {window}-day volatility')
valid = feature_parts[0].notna().all(axis=1)
for part in feature_parts[1:]:
    valid &= part.notna().all(axis=1)
dates = returns.index[valid]
feature_array = np.stack([part.loc[dates].to_numpy() for part in feature_parts], axis=2)
cost_array = -returns.loc[dates].to_numpy()
train_mask = dates <= pd.Timestamp('2021-12-31')
validation_mask = (dates >= pd.Timestamp('2022-01-01')) & (dates <= pd.Timestamp('2023-12-31'))
test_mask = dates >= pd.Timestamp('2024-01-01')
feature_mean = feature_array[train_mask].mean(axis=(0, 1), keepdims=True)
feature_scale = feature_array[train_mask].std(axis=(0, 1), ddof=1, keepdims=True)
standardized = (feature_array - feature_mean) / feature_scale
design = np.concatenate([np.ones((*standardized.shape[:2], 1)), standardized], axis=2)
vertices = np.zeros((1820, 16))
members = []
for row, selected in enumerate(combinations(range(16), 4)):
    vertices[row, list(selected)] = 0.25
    members.append(selected)
assert len(members) == 1820
assert np.allclose(vertices.sum(axis=1), 1)
provenance = pd.DataFrame([{'Item': 'Public source', 'Value': 'Kenneth R. French Data Library, 49 Industry Portfolios Daily'}, {'Item': 'Stored archive', 'Value': DATA_PATH.name}, {'Item': 'Industries retained', 'Value': len(industries)}, {'Item': 'Download during execution', 'Value': 'None'}, {'Item': 'Simulated observations', 'Value': 'None'}])
save_table(provenance, 'decision_public_data_provenance', 'Public data provenance for the decision experiment.', 'tab:chapter28-decision-data-provenance', fit=True)
raw_periods = [('Training', '2019-01-01', '2021-12-31'), ('Validation', '2022-01-01', '2023-12-31'), ('Final test', '2024-01-01', '2025-12-31')]
period_rows = []
for name, start, end in raw_periods:
    raw_count = int(((public['Date'] >= start) & (public['Date'] <= end)).sum())
    usable_count = int(((dates >= start) & (dates <= end)).sum())
    period_rows.append({'Period': name, 'Start': start, 'End': end, 'Raw observations': raw_count, 'Usable decisions': usable_count})
period_table = pd.DataFrame(period_rows)
assert period_table['Raw observations'].tolist() == [757, 501, 502]
save_table(period_table, 'decision_chronological_periods', 'Fixed chronological periods before and after feature losses.', 'tab:chapter28-decision-periods')
feature_table = pd.DataFrame({'Feature': feature_names, 'Window': [5, 20, 60, 5, 20, 60], 'Latest permitted input': ['Day before decision'] * 6, 'Training standardization only': [True] * 6})
save_table(feature_table, 'decision_feature_definitions', 'Past-return features used by both linear rules.', 'tab:chapter28-feature-definitions', fit=True)
timing_rows = []
for window in [5, 20, 60]:
    first_position = returns.index.get_loc(dates[0])
    latest_input = returns.index[first_position - 1]
    earliest_input = returns.index[first_position - window]
    timing_rows.append({'Window': window, 'First target date': dates[0], 'Earliest input date': earliest_input, 'Latest input date': latest_input, 'Latest input precedes target': latest_input < dates[0]})
timing_table = pd.DataFrame(timing_rows)
assert timing_table['Latest input precedes target'].all()
save_table(timing_table, 'decision_feature_timing_checks', 'Direct checks that every first feature window ends before its target.', 'tab:chapter28-feature-timing', fit=True)
feasible_summary = pd.DataFrame([{'Quantity': 'Assets', 'Value': 16, 'Check': True}, {'Quantity': 'Assets held per vertex', 'Value': 4, 'Check': True}, {'Quantity': 'Weight per selected asset', 'Value': 0.25, 'Check': True}, {'Quantity': 'Enumerated vertices', 'Value': len(vertices), 'Check': len(vertices) == 1820}, {'Quantity': 'Smallest vertex total', 'Value': vertices.sum(axis=1).min(), 'Check': np.allclose(vertices.sum(axis=1), 1)}, {'Quantity': 'Largest vertex total', 'Value': vertices.sum(axis=1).max(), 'Check': np.allclose(vertices.sum(axis=1), 1)}])
save_table(feasible_summary, 'decision_feasible_set_summary', 'Checks for the supplied four-industry portfolio vertices.', 'tab:chapter28-feasible-set')
timeline = period_table.copy()
timeline['Start date'] = pd.to_datetime(timeline['Start'])
timeline['End date'] = pd.to_datetime(timeline['End'])
timeline[['Period', 'Start', 'End', 'Raw observations', 'Usable decisions']].to_csv(FIGURE_DATA / 'decision_timeline.csv', index=False)
figure, axis = plt.subplots(figsize=(10.2, 3.8), constrained_layout=True)
colors = [PALETTE['cyan'], PALETTE['amber'], PALETTE['green']]
for index, row in timeline.iterrows():
    axis.barh(0, (row['End date'] - row['Start date']).days + 1, left=row['Start date'], height=0.44, color=colors[index], label=f"{row['Period']} ({row['Usable decisions']})")
axis.set_yticks([])
axis.set_xlabel('Calendar date')
axis.set_title('Training, Validation, and Final Test Remain in Time Order')
axis.legend(ncol=3, loc='upper center')
finish_axis(axis, grid=False)
save_figure(figure, 'decision_timeline')


,Item,Value
0,Public source,"Kenneth R. French Data Library, 49 Industry Po..."
1,Stored archive,49_Industry_Portfolios_daily_CSV.zip
2,Industries retained,16
3,Download during execution,None
4,Simulated observations,None


,Period,Start,End,Raw observations,Usable decisions
0,Training,2019-01-01,2021-12-31,757,697
1,Validation,2022-01-01,2023-12-31,501,501
2,Final test,2024-01-01,2025-12-31,502,502


,Feature,Window,Latest permitted input,Training standardization only
0,Past 5-day mean,5,Day before decision,Pass
1,Past 20-day mean,20,Day before decision,Pass
2,Past 60-day mean,60,Day before decision,Pass
3,Past 5-day volatility,5,Day before decision,Pass
4,Past 20-day volatility,20,Day before decision,Pass
5,Past 60-day volatility,60,Day before decision,Pass


,Window,First target date,Earliest input date,Latest input date,Latest input precedes target
0,5,2019-03-29,2019-03-22,2019-03-28,Pass
1,20,2019-03-29,2019-03-01,2019-03-28,Pass
2,60,2019-03-29,2019-01-02,2019-03-28,Pass


,Quantity,Value,Check
0,Assets,16,Pass
1,Assets held per vertex,4,Pass
2,Weight per selected asset,0.25,Pass
3,Enumerated vertices,"1,820",Pass
4,Smallest vertex total,1,Pass
5,Largest vertex total,1,Pass


<Figure size 1020x380 with 1 Axes>

## Two Linear Prediction Rules with the Same Size

Both rules use the same shared coefficient vector $\beta\in\mathbb R^{7}$:

$$
\widehat c_{i,t}=x_{i,t}^{\mathsf T}\beta.
$$

The first rule minimizes the average squared error over sampled industry costs. The second minimizes the average SPO+ loss over sampled dates. Both start at zero, use the same eighty update steps, the same batch size, and the same learning rate. Only the training loss changes. Validation and final test observations never update either rule.

In [2]:
torch.set_num_threads(1)
train_x = torch.tensor(design[train_mask], dtype=torch.float64)
train_c = torch.tensor(cost_array[train_mask], dtype=torch.float64)
vertex_tensor = torch.tensor(vertices, dtype=torch.float64)
iterations = 80
batch_size = 32
learning_rate = 0.02
seed = 2802

def train_rule(loss_name):
    beta = torch.zeros(design.shape[2], dtype=torch.float64, requires_grad=True)
    optimizer = torch.optim.Adam([beta], lr=learning_rate)
    generator = torch.Generator().manual_seed(seed)
    rows = []
    for iteration in range(iterations):
        indices = torch.randint(0, len(train_x), (batch_size,), generator=generator)
        prediction = train_x[indices] @ beta
        truth = train_c[indices]
        if loss_name == "Squared error":
            loss = torch.mean((prediction - truth) ** 2)
        else:
            loss = spo_plus_torch(prediction, truth, vertex_tensor).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        rows.append({"Iteration": iteration + 1, "Rule": loss_name, "Training loss": float(loss.detach())})
    return beta.detach().numpy(), pd.DataFrame(rows)

beta_mse, history_mse = train_rule("Squared error")
beta_spo, history_spo = train_rule("SPO+")
history = pd.concat([history_mse, history_spo], ignore_index=True)
assert np.isfinite(beta_mse).all() and np.isfinite(beta_spo).all()

settings = pd.DataFrame([
    {"Setting": "Shared coefficients per rule", "Value": len(beta_mse)},
    {"Setting": "Update steps", "Value": iterations},
    {"Setting": "Batch size", "Value": batch_size},
    {"Setting": "Learning rate", "Value": learning_rate},
    {"Setting": "Random seed", "Value": seed},
    {"Setting": "Feasible vertices", "Value": len(vertices)},
    {"Setting": "Final test used in training", "Value": "No"},
])
save_table(settings, "decision_training_settings", "Fixed settings shared by the two linear training runs.", "tab:chapter28-training-settings")

loss_summary = history.groupby("Rule")["Training loss"].agg(["first", "last", "min"]).reset_index().rename(columns={"first": "First loss", "last": "Last loss", "min": "Smallest saved loss"})
save_table(loss_summary, "decision_training_loss_summary", "Saved training loss summaries for the two rules.", "tab:chapter28-training-loss-summary")

parameters = pd.DataFrame({
    "Feature": ["Intercept", *feature_names],
    "Squared error coefficient": beta_mse,
    "SPO+ coefficient": beta_spo,
})
save_table(parameters, "decision_linear_coefficients", "Shared linear coefficients after the fixed training runs.", "tab:chapter28-linear-coefficients", fit=True)

history.to_csv(FIGURE_DATA / "decision_training_losses.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.4, 4.6), constrained_layout=True)
for axis, name, color in zip(axes, ["Squared error", "SPO+"], [PALETTE["cyan"], PALETTE["amber"]]):
    part = history.loc[history["Rule"] == name]
    axis.plot(part["Iteration"], part["Training loss"], color=color, linewidth=1.9)
    axis.set_title(name)
    axis.set_xlabel("Update step")
    axis.set_ylabel("Sampled training loss")
    finish_axis(axis)
save_figure(figure, "decision_training_losses")

,Setting,Value
0,Shared coefficients per rule,7
1,Update steps,80
2,Batch size,32
3,Learning rate,0.02
4,Random seed,"2,802"
5,Feasible vertices,"1,820"
6,Final test used in training,No


,Rule,First loss,Last loss,Smallest saved loss
0,SPO+,2.315781,2.35561,2.122893
1,Squared error,1.66682,5.423882,1.386535


,Feature,Squared error coefficient,SPO+ coefficient
0,Intercept,-0.039774,0
1,Past 5-day mean,-0.088551,0.10558
2,Past 20-day mean,0.044946,0.028659
3,Past 60-day mean,0.068835,0.015084
4,Past 5-day volatility,0.158182,-0.041579
5,Past 20-day volatility,-0.020465,0.013928
6,Past 60-day volatility,-0.088126,-0.044133


<Figure size 1040x460 with 2 Axes>

## Chronological Evaluation

For a rule $m$ on date $t$, let $\widehat w_{m,t}=w^{*}(\widehat c_{m,t})$. Its observed regret is

$$
R_{m,t}
=
c_t^{\mathsf T}\widehat w_{m,t}
-
c_t^{\mathsf T}w^{*}(c_t).
$$

Here, $R_{m,t}$ compares the selected vertex with the best supplied vertex after the next return is observed. It is nonnegative by construction. Turnover is one half of the sum of absolute weight changes between successive selected vertices. These are descriptive results for the declared split.

In [3]:
def evaluate_rule(beta, mask, name):
    prediction = design[mask] @ beta
    truth = cost_array[mask]
    predicted_index = np.argmin(prediction @ vertices.T, axis=1)
    true_index = np.argmin(truth @ vertices.T, axis=1)
    selected = vertices[predicted_index]
    optimal = vertices[true_index]
    selected_cost = np.einsum("ij,ij->i", truth, selected)
    optimal_cost = np.einsum("ij,ij->i", truth, optimal)
    regret = selected_cost - optimal_cost
    turnover = np.r_[np.nan, 0.5 * np.abs(np.diff(selected, axis=0)).sum(axis=1)]
    assert regret.min() >= -1e-10
    return {
        "name": name,
        "dates": dates[mask],
        "prediction": prediction,
        "truth": truth,
        "predicted_index": predicted_index,
        "true_index": true_index,
        "selected": selected,
        "selected_cost": selected_cost,
        "optimal_cost": optimal_cost,
        "regret": regret,
        "turnover": turnover,
    }

def metric_row(result, period):
    return {
        "Period": period,
        "Rule": result["name"],
        "Mean squared error": float(np.mean((result["prediction"] - result["truth"]) ** 2)),
        "Mean decision regret": float(result["regret"].mean()),
        "Mean selected cost": float(result["selected_cost"].mean()),
        "Mean best vertex cost": float(result["optimal_cost"].mean()),
        "Exact decision share": float(np.mean(result["predicted_index"] == result["true_index"])),
        "Mean turnover": float(np.nanmean(result["turnover"])),
    }

validation_results = [evaluate_rule(beta_mse, validation_mask, "Squared error"), evaluate_rule(beta_spo, validation_mask, "SPO+")]
test_results = [evaluate_rule(beta_mse, test_mask, "Squared error"), evaluate_rule(beta_spo, test_mask, "SPO+")]
validation_metrics = pd.DataFrame([metric_row(result, "Validation") for result in validation_results])
test_metrics = pd.DataFrame([metric_row(result, "Final test") for result in test_results])
save_table(validation_metrics, "decision_validation_metrics", "Fixed-rule diagnostics in the validation period.", "tab:chapter28-validation-metrics", fit=True)
save_table(test_metrics, "decision_final_test_metrics", "Untouched final-period diagnostics for the two fixed rules.", "tab:chapter28-test-metrics", fit=True)

equal_cost = cost_array[test_mask].mean(axis=1)
benchmark = pd.DataFrame([
    {"Decision": "Equal weight over sixteen industries", "Mean observed cost": equal_cost.mean(), "Mean turnover": 0.0, "Inside four-industry feasible set": False},
    {"Decision": "Squared error selected vertices", "Mean observed cost": test_results[0]["selected_cost"].mean(), "Mean turnover": np.nanmean(test_results[0]["turnover"]), "Inside four-industry feasible set": True},
    {"Decision": "SPO+ selected vertices", "Mean observed cost": test_results[1]["selected_cost"].mean(), "Mean turnover": np.nanmean(test_results[1]["turnover"]), "Inside four-industry feasible set": True},
])
save_table(benchmark, "decision_equal_weight_context", "Equal weight is reported as an outside reference, not as a vertex-regret comparison.", "tab:chapter28-equal-weight-context", fit=True)

all_metrics = pd.concat([validation_metrics, test_metrics], ignore_index=True)
all_metrics.to_csv(FIGURE_DATA / "decision_metric_comparison.csv", index=False)
figure, axes = plt.subplots(1, 2, figsize=(10.4, 4.7), constrained_layout=True)
for index, metric in enumerate(["Mean squared error", "Mean decision regret"]):
    pivot = all_metrics.pivot(index="Period", columns="Rule", values=metric)
    pivot.plot.bar(ax=axes[index], color=[PALETTE["cyan"], PALETTE["amber"]], rot=0)
    axes[index].set_title(metric)
    axes[index].set_xlabel("")
    finish_axis(axes[index])
save_figure(figure, "decision_metric_comparison")

cumulative_rows = []
for result in test_results:
    cumulative = np.cumsum(result["regret"])
    cumulative_rows.extend({"Date": date, "Rule": result["name"], "Cumulative regret": value} for date, value in zip(result["dates"], cumulative))
cumulative_table = pd.DataFrame(cumulative_rows)
cumulative_table.to_csv(FIGURE_DATA / "decision_cumulative_regret.csv", index=False)
figure, axis = plt.subplots(figsize=(10.0, 5.1), constrained_layout=True)
for name, color in [("Squared error", PALETTE["cyan"]), ("SPO+", PALETTE["amber"])]:
    part = cumulative_table.loc[cumulative_table["Rule"] == name]
    axis.plot(pd.to_datetime(part["Date"]), part["Cumulative regret"], color=color, linewidth=2.0, label=name)
axis.set_xlabel("Final test date")
axis.set_ylabel("Cumulative decision regret")
axis.set_title("Final Test Regret Accumulates Differently Across Fixed Rules")
axis.legend()
finish_axis(axis)
save_figure(figure, "decision_cumulative_regret")

turnover_rows = []
for result in test_results:
    turnover_rows.extend({"Date": date, "Rule": result["name"], "Turnover": value} for date, value in zip(result["dates"], result["turnover"]) if np.isfinite(value))
turnover_table = pd.DataFrame(turnover_rows)
turnover_table.to_csv(FIGURE_DATA / "decision_turnover.csv", index=False)
figure, axis = plt.subplots(figsize=(10.0, 5.1), constrained_layout=True)
for name, color in [("Squared error", PALETTE["cyan"]), ("SPO+", PALETTE["amber"])]:
    part = turnover_table.loc[turnover_table["Rule"] == name]
    rolling = part.set_index(pd.to_datetime(part["Date"]))["Turnover"].rolling(20, min_periods=1).mean()
    axis.plot(rolling.index, rolling, color=color, linewidth=1.9, label=name)
axis.set_xlabel("Final test date")
axis.set_ylabel("Twenty-day mean turnover")
axis.set_title("Selected Decisions Change Through the Final Period")
axis.legend()
finish_axis(axis)
save_figure(figure, "decision_turnover")

frequency_rows = []
for result in test_results:
    frequencies = result["selected"].mean(axis=0) * 4
    frequency_rows.extend({"Industry": industry, "Rule": result["name"], "Selection share": value} for industry, value in zip(industries, frequencies))
frequency_table = pd.DataFrame(frequency_rows)
save_table(frequency_table, "decision_industry_selection_shares", "Final-period share of dates on which each industry is selected.", "tab:chapter28-selection-shares", fit=True)
save_table(frequency_table.loc[frequency_table["Rule"] == "Squared error"].reset_index(drop=True), "decision_industry_selection_shares_mse", "Final-period industry selection shares for the squared-error rule.", "tab:chapter28-selection-shares-mse")
save_table(frequency_table.loc[frequency_table["Rule"] == "SPO+"].reset_index(drop=True), "decision_industry_selection_shares_spo", "Final-period industry selection shares for the SPO+ rule.", "tab:chapter28-selection-shares-spo")
frequency_table.to_csv(FIGURE_DATA / "decision_selection_frequencies.csv", index=False)
figure, axis = plt.subplots(figsize=(11.0, 5.2), constrained_layout=True)
x = np.arange(len(industries))
for offset, name, color in [(-0.19, "Squared error", PALETTE["cyan"]), (0.19, "SPO+", PALETTE["amber"])]:
    part = frequency_table.loc[frequency_table["Rule"] == name].set_index("Industry").loc[industries]
    axis.bar(x + offset, part["Selection share"], width=0.38, color=color, label=name)
axis.set_xticks(x, industries, rotation=55, ha="right")
axis.set_ylabel("Selection share")
axis.set_title("Industry Selection Frequencies in the Final Period")
axis.legend()
finish_axis(axis)
save_figure(figure, "decision_selection_frequencies")

decision_summary_rows = []
for result in test_results:
    counts = pd.Series(result["predicted_index"]).value_counts()
    for vertex_index, count in counts.head(5).items():
        decision_summary_rows.append({"Rule": result["name"], "Vertex": int(vertex_index), "Industries": ", ".join(industries[index] for index in members[int(vertex_index)]), "Selected dates": int(count)})
decision_summary = pd.DataFrame(decision_summary_rows)
save_table(decision_summary, "decision_most_common_vertices", "Five most common final-period vertices for each rule.", "tab:chapter28-common-vertices", fit=True)

limits = pd.DataFrame([
    {"Question": "Are all features based on earlier returns?", "Answered": True, "Boundary": "Direct timing checks"},
    {"Question": "Are both rules the same size?", "Answered": True, "Boundary": "Seven shared coefficients each"},
    {"Question": "Was the final test used to train or tune?", "Answered": False, "Boundary": "Untouched comparison"},
    {"Question": "Does one split establish general superiority?", "Answered": False, "Boundary": "No general claim"},
    {"Question": "Is the finite oracle a general solver?", "Answered": False, "Boundary": "Only supplied vertices"},
])
save_table(limits, "decision_public_study_boundaries", "Questions answered and not answered by the public decision study.", "tab:chapter28-decision-boundaries", fit=True)

,Period,Rule,Mean squared error,Mean decision regret,Mean selected cost,Mean best vertex cost,Exact decision share,Mean turnover
0,Validation,Squared error,2.508977,1.209067,0.03475,-1.174316,0.003992,0.282
1,Validation,SPO+,2.510452,1.182146,0.007829,-1.174316,0.005988,0.257


,Period,Rule,Mean squared error,Mean decision regret,Mean selected cost,Mean best vertex cost,Exact decision share,Mean turnover
0,Final test,Squared error,1.848613,1.085533,-0.078013,-1.163546,0,0.283433
1,Final test,SPO+,1.844269,1.113217,-0.050329,-1.163546,0.005976,0.279441


,Decision,Mean observed cost,Mean turnover,Inside four-industry feasible set
0,Equal weight over sixteen industries,-0.032652,0,Fail
1,Squared error selected vertices,-0.078013,0.283433,Pass
2,SPO+ selected vertices,-0.050329,0.279441,Pass


<Figure size 1040x470 with 2 Axes>

<Figure size 1000x510 with 1 Axes>

...

,Industry,Rule,Selection share
0,Agric,SPO+,0.262948
1,Food,SPO+,0.171315
2,Soda,SPO+,0.103586
3,Beer,SPO+,0.312749
4,Smoke,SPO+,0.239044
5,Toys,SPO+,0.342629
6,Fun,SPO+,0.256972
7,Books,SPO+,0.175299
8,Hshld,SPO+,0.145418
9,Clths,SPO+,0.501992


<Figure size 1100x520 with 1 Axes>

,Rule,Vertex,Industries,Selected dates
0,Squared error,"1,537","Toys, Books, Clths, Rubbr",11
1,Squared error,"1,587","Toys, Clths, Chems, Rubbr",9
2,Squared error,300,"Agric, Toys, Books, Clths",7
3,Squared error,576,"Food, Beer, Hshld, Rubbr",6
4,Squared error,"1,589","Toys, Clths, Rubbr, Txtls",6
5,SPO+,"1,588","Toys, Clths, Chems, Txtls",17
6,SPO+,"1,589","Toys, Clths, Rubbr, Txtls",12
7,SPO+,"1,579","Toys, Clths, Hlth, Txtls",9
8,SPO+,"1,509","Toys, Fun, Clths, Rubbr",8
9,SPO+,319,"Agric, Toys, Clths, Txtls",8


,Question,Answered,Boundary
0,Are all features based on earlier returns?,Pass,Direct timing checks
1,Are both rules the same size?,Pass,Seven shared coefficients each
2,Was the final test used to train or tune?,Fail,Untouched comparison
3,Does one split establish general superiority?,Fail,No general claim
4,Is the finite oracle a general solver?,Fail,Only supplied vertices


,Question,Answered,Boundary
0,Are all features based on earlier returns?,Pass,Direct timing checks
1,Are both rules the same size?,Pass,Seven shared coefficients each
2,Was the final test used to train or tune?,Fail,Untouched comparison
3,Does one split establish general superiority?,Fail,No general claim
4,Is the finite oracle a general solver?,Fail,Only supplied vertices
